# Lesson 1: Prompt Evaluation with Gemini (Companion Notebook)

This notebook mirrors the Coursera **"Evaluating Prompt Performance"** series using **Google Gemini (`gemini-3.1-flash-lite`)** and the modern `google-genai` SDK.

### Core Evaluation Pipeline Architecture:
1. **Synthetic Dataset Generation**: Generating AWS code/JSON/Regex test cases with criteria.
2. **Prompt Execution**: Running candidate prompts against test cases.
3. **Model-Based Grading (LLM-as-a-Judge)**: Multi-attribute scoring (`strengths`, `weaknesses`, `reasoning`, `score`).
4. **Code-Based Grading (Deterministic Validation)**: Abstract Syntax Trees (`ast`), JSON parsers (`json`), and Regex engines (`re`).
5. **Hybrid Composite Scoring**: Combining model judgment and deterministic validation: `(model_score + syntax_score) / 2`.

### Anthropic Claude vs. Google Gemini Design Patterns:
- **Anthropic Claude**: Structured output via assistant prefilling (`messages.append({"role": "assistant", "content": "```json"})`) and stop sequences `["```"]`.
- **Google Gemini**: Structured output via native `response_mime_type="application/json"` and Pydantic schemas in `GenerateContentConfig`.

In [ ]:
# 1. Load environment variables and initialize Gemini client
import os
import json
import re
import ast
from statistics import mean
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

load_dotenv()

client = genai.Client()
MODEL_ID = os.getenv("GEMINI_MODEL_ID", "gemini-3.1-flash-lite")
print(f"Initialized Google GenAI client with model: {MODEL_ID}")

## 1. Generating the Evaluation Dataset

We ask Gemini to generate synthetic test vectors for AWS-related tasks. Each item includes:
- `task`: Description of what AWS task needs to be solved
- `format`: Expected output type (`json`, `python`, or `regex`)
- `solution_criteria`: Specific requirements the model grader should verify

In [ ]:
# Schema for evaluation dataset items
class EvalItem(BaseModel):
    task: str = Field(description="Description of AWS task requiring Python, JSON, or Regex")
    format: str = Field(description="Expected format: 'json', 'python', or 'regex'")
    solution_criteria: str = Field(description="Criteria needed for a correct and complete solution")


def generate_dataset(num_samples: int = 3):
    prompt = f"""
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks.

Focus on tasks that:
- Can be solved by writing a single Python function, a single JSON configuration, or a regular expression.
- Do not require large amounts of boilerplate.
- Have clear, unambiguous success criteria.

Please generate {num_samples} objects.
"""
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=list[EvalItem],
            temperature=0.7,
        ),
    )
    return json.loads(response.text)


# If dataset.json exists, load it; otherwise generate a new one
if os.path.exists("dataset.json"):
    with open("dataset.json", "r") as f:
        dataset = json.load(f)
    print(f"Loaded {len(dataset)} items from dataset.json")
else:
    dataset = generate_dataset(num_samples=3)
    with open("dataset.json", "w") as f:
        json.dump(dataset, f, indent=2)
    print(f"Generated and saved {len(dataset)} items to dataset.json")

print(json.dumps(dataset[:2], indent=2))

## 2. Model-Based Grading (LLM-as-a-Judge)

We define an expert AWS reviewer prompt that consumes the `<task>`, `<solution>`, and `<criteria>`, returning structured evaluation feedback:
- `strengths`: List of strengths
- `weaknesses`: List of areas for improvement
- `reasoning`: Overall assessment
- `score`: Calibrated numerical score (1-10)

In [ ]:
# Schema for Model-Based Evaluation output
class ModelEvaluation(BaseModel):
    strengths: list[str] = Field(description="1 to 3 key strengths of the solution")
    weaknesses: list[str] = Field(description="1 to 3 key weaknesses or missed criteria")
    reasoning: str = Field(description="Concise explanation of overall quality")
    score: float = Field(description="Calibrated score between 1.0 and 10.0")


def grade_by_model(test_case: dict, output: str) -> dict:
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case['task']}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case.get('solution_criteria', 'Must be syntactically correct and fully solve the task.')}
</criteria>

Evaluate thoroughly against the criteria and output structured JSON assessment.
"""
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=eval_prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=ModelEvaluation,
            temperature=0.2,
        ),
    )
    return json.loads(response.text)

## 3. Code-Based Syntax Validation

Deterministic verification using native Python runtimes:
- **JSON**: Parsed via `json.loads`
- **Python**: Parsed into Abstract Syntax Tree via `ast.parse`
- **Regex**: Compiled via `re.compile`

Returns `10` if valid, `0` if syntax error occurs.

In [ ]:
def validate_json(text: str) -> float:
    cleaned = text.strip()
    if cleaned.startswith("```"): cleaned = re.sub(r"^```(?:json)?\n?|```$", "", cleaned).strip()
    try:
        json.loads(cleaned)
        return 10.0
    except (json.JSONDecodeError, ValueError):
        return 0.0


def validate_python(text: str) -> float:
    cleaned = text.strip()
    if cleaned.startswith("```"): cleaned = re.sub(r"^```(?:python)?\n?|```$", "", cleaned).strip()
    try:
        ast.parse(cleaned)
        return 10.0
    except SyntaxError:
        return 0.0


def validate_regex(text: str) -> float:
    cleaned = text.strip()
    if cleaned.startswith("```"): cleaned = re.sub(r"^```(?:regex)?\n?|```$", "", cleaned).strip()
    try:
        re.compile(cleaned)
        return 10.0
    except re.error:
        return 0.0


def grade_syntax(response: str, test_case: dict) -> float:
    fmt = test_case.get("format", "").lower()
    if fmt == "json":
        return validate_json(response)
    elif fmt == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

## 4. Evaluation Runner & Composite Scoring

We tie the candidate prompt, model grader, and syntax validator together into `run_test_case` and `run_eval`.

In [ ]:
def run_prompt(test_case: dict) -> str:
    """Passes the test case into the model under test."""
    prompt = f"""
Please solve the following task:

{test_case['task']}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.2,
        ),
    )
    return response.text


def run_test_case(test_case: dict) -> dict:
    """Executes a single test case, applies both graders, and computes composite score."""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)
    composite_score = (model_score + syntax_score) / 2.0

    return {
        "task": test_case["task"],
        "format": test_case["format"],
        "output": output,
        "model_score": model_score,
        "syntax_score": syntax_score,
        "composite_score": composite_score,
        "strengths": model_grade["strengths"],
        "weaknesses": model_grade["weaknesses"],
        "reasoning": reasoning,
    }


def run_eval(dataset: list[dict]) -> list[dict]:
    """Runs the evaluation suite over all test cases and reports summary statistics."""
    results = []
    print(f"\n{'='*60}")
    print(f"Starting Evaluation on {len(dataset)} Test Cases...")
    print(f"{'='*60}\n")

    for i, test_case in enumerate(dataset, 1):
        print(f"[{i}/{len(dataset)}] Evaluating: {test_case['task'][:50]}...")
        result = run_test_case(test_case)
        results.append(result)
        print(f"   -> Model: {result['model_score']}/10 | Syntax: {result['syntax_score']}/10 | Composite: {result['composite_score']:.1f}/10")

    avg_model = mean([r["model_score"] for r in results])
    avg_syntax = mean([r["syntax_score"] for r in results])
    avg_composite = mean([r["composite_score"] for r in results])

    print(f"\n{'='*60}")
    print("EVALUATION COMPLETE")
    print(f"Average Model Score:     {avg_model:.2f} / 10")
    print(f"Average Syntax Score:    {avg_syntax:.2f} / 10")
    print(f"Average Composite Score: {avg_composite:.2f} / 10")
    print(f"{'='*60}\n")

    return results

## 5. Execute Evaluation Suite & Inspect Results

In [ ]:
# Execute evaluation on our dataset
eval_results = run_eval(dataset)

# Display full detailed report
print(json.dumps(eval_results, indent=2))